In [1]:
import requests
import json
import pandas as pd
from bs4 import BeautifulSoup

# Read products_id csv file
df = pd.read_csv("products-0-200000.csv")  #

headers = {"User-Agent": "Mozilla/5.0"}
products = []

success_count = 0
error_count = 0

# Open log file to write messages
log_file = open("log.txt", "w", encoding="utf-8")

for product_id in df["id"]:
    url = f"https://api.tiki.vn/product-detail/api/v1/products/{product_id}"
    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        try:
            data = response.json()
        except json.JSONDecodeError:
            msg = f"[ERROR] Can not parse JSON for product {product_id}\n"
            print(msg.strip())
            log_file.write(msg)
            error_count += 1
            continue

        # html description -> clean text
        raw_description = data.get("description", "")
        soup = BeautifulSoup(raw_description, "html.parser")
        clean_description = soup.get_text(separator=" ", strip=True)

        product_info = {
            "id": data.get("id"),
            "name": data.get("name"),
            "url_key": data.get("url_key"),
            "price": data.get("price"),
            "description": clean_description,
            "images": [img.get("base_url") for img in data.get("images", [])]
        }

        products.append(product_info)
        success_count += 1
        msg = f"[OK] GET JSON successfully for product {product_id}\n"
        print(msg.strip())
        log_file.write(msg)
    else:
        msg = f"[ERROR] Failed {response.status_code} to GET product {product_id}\n"
        print(msg.strip())
        log_file.write(msg)
        error_count += 1

# Save with "data": array of products
output = {"data": products}

with open("products_full.json", "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=4)

# Summary
summary = (
    f"=== Kết quả ===\n"
    f"Numbers of product GET succesfully: {success_count}\n"
    f"Numbers of ERROR GET: {error_count}\n"
    f"Saved to products.json\n"
)
print(summary.strip())
log_file.write(summary)

log_file.close()


[OK] GET JSON successfully for product 1391347
[OK] GET JSON successfully for product 74897599
[OK] GET JSON successfully for product 154155413
[OK] GET JSON successfully for product 253117062
[OK] GET JSON successfully for product 130978358
[OK] GET JSON successfully for product 214009046
[OK] GET JSON successfully for product 171618108
[OK] GET JSON successfully for product 179970479
[OK] GET JSON successfully for product 139457837
[OK] GET JSON successfully for product 197334787
[OK] GET JSON successfully for product 214008432
[OK] GET JSON successfully for product 213678661
[OK] GET JSON successfully for product 138083218
[OK] GET JSON successfully for product 75331097
[OK] GET JSON successfully for product 139457787
[OK] GET JSON successfully for product 146457002
[OK] GET JSON successfully for product 178596691
[OK] GET JSON successfully for product 191463903
[OK] GET JSON successfully for product 167750518
[OK] GET JSON successfully for product 170337343
[OK] GET JSON successful

KeyboardInterrupt: 